# An-Ra V4 — core-vnext TPU training (post-20k path)
#
# Fail-closed training on verified token packs with a WSD LR schedule.
# Design doc: docs/planning/POST_20K_AGI_PATH.md
#
# Before Run All:
#   1. Accelerator -> TPU v5e-8, complete verification, restart if needed.
#   2. Attach TWO Kaggle datasets:
#        a) checkpoint dataset  -> contains anra-v4-current-full-resume.pt (step 20000+)
#        b) token-pack dataset  -> a .tar.gz containing manifest.json + train/*.npy
#   3. Edit CONFIG in cell 4 only.
#
# This notebook REFUSES to train on unverified data. If you see
# 'REFUSING TO TRAIN', the attached pack failed verification - do not bypass.


In [ ]:
# 1. TPU runtime preflight — fail closed.
import importlib.util, os, sys
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError('TPU not attached. Settings > Accelerator > TPU v5e-8, verify, restart, Run All.')
import torch
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except (ImportError, OSError) as exc:
    raise RuntimeError(f'Unusable XLA runtime: {exc}') from exc
if int(xr.global_device_count()) != 8:
    raise RuntimeError(f'Need all 8 cores of v5e-8, got {xr.global_device_count()}. Reconnect.')
print({'device': str(xm.xla_device()), 'cores': xr.global_device_count(), 'torch': torch.__version__})

In [ ]:
# 2. Clone the exact training branch and record its commit for provenance.
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
sys.path.insert(0, str(REPO))
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print(json.dumps({'repo': str(REPO), 'commit': SOURCE_COMMIT[:12], 'branch': REPO_REF}))

In [ ]:
# 3. Locate checkpoint + token pack. The pack MUST carry a valid manifest.
import hashlib, json, tarfile
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as bundle:
        root = destination.resolve()
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        bundle.extractall(destination)

def find_checkpoint():
    best, best_step = None, -1
    for p in INPUT_ROOT.rglob('*.pt'):
        try:
            payload = torch.load(p, map_location='cpu', weights_only=True)
            step = int(payload.get('global_step', -1)) if isinstance(payload, dict) else -1
            del payload
        except Exception:
            continue
        if step > best_step:
            best, best_step = p, step
    if best is None or best_step < 1:
        raise FileNotFoundError('No readable full-resume .pt checkpoint found in /kaggle/input.')
    return best, best_step

def find_pack():
    """Accept ONLY an archive that yields manifest.json + train/*.npy."""
    archives = sorted(INPUT_ROOT.rglob('*.tar.gz'))
    if not archives:
        raise FileNotFoundError(
            'No *.tar.gz token pack found. This notebook trains ONLY on '
            'manifest-verified packs; plain text files are refused by design.'
        )
    dest = Path('/kaggle/working/pack')
    safe_extract(archives[0], dest)
    manifest = dest / 'manifest.json'
    train_dir = dest / 'train'
    if not manifest.is_file() or not train_dir.is_dir():
        raise RuntimeError(
            'Archive does not contain manifest.json + train/*.npy. '
            'Build the pack with training/pack_verify.build_manifest first.'
        )
    return dest

CHECKPOINT, RESUME_STEP = find_checkpoint()
PACK_ROOT = find_pack()
print(json.dumps({
    'checkpoint': str(CHECKPOINT),
    'resume_step': RESUME_STEP,
    'checkpoint_sha256': sha256_file(CHECKPOINT)[:16],
    'pack_root': str(PACK_ROOT),
}, indent=2))

In [ ]:
# 4. CONFIG — edit only these values.
RESUME_FROM   = str(CHECKPOINT)      # step >= 20000 recommended (see POST_20K_AGI_PATH.md)
PACK_DIR      = str(PACK_ROOT)
OUTPUT_CKPT   = '/kaggle/working/anra-v4-trained.pt'
MAX_MINUTES   = 430                  # leave headroom before Kaggle's 9h limit
TOKEN_BUDGET  = 330_000_000          # unique tokens in THIS pack; decay lands at its end
BATCH_SIZE    = 1
GRAD_ACCUM    = 8                    # 1*8*8*2048 = 131,072 tokens/step
LEARNING_RATE = 2e-4                 # WSD: 2% warmup -> stable -> decay to 10%
MIN_LR_RATIO  = 0.10
SAVE_INTERVAL = 400
LOG_INTERVAL  = 10
SEED          = 1301
print(json.dumps({k: v for k, v in globals().items() if k.isupper()}, indent=2, default=str))

In [ ]:
# 5. Verify the pack FAIL-CLOSED before any GPU-hour is spent.
from pathlib import Path
from training.pack_verify import PackVerificationError, verify_pack
try:
    pack = verify_pack(Path(PACK_DIR))
except PackVerificationError as exc:
    raise SystemExit(f'REFUSING TO TRAIN: {exc}') from exc
print(f'PACK VERIFIED: {len(pack.shard_paths)} shards | '
      f'{pack.total_tokens:,} tokens | ~{pack.total_windows:,} unique windows')

In [ ]:
# 6. Sanity-load the model + checkpoint identity on CPU first.
from anra_core.checkpoint import load_core_checkpoint
model, payload, identity = load_core_checkpoint(RESUME_FROM)
tok = __import__('anra_core.tokenizer', fromlist=['V4Tokenizer']).V4Tokenizer.load_canonical()
assert identity.tokenizer_contract_verified, 'tokenizer contract must be verified'
params = sum(p.numel() for p in model.parameters())
print(json.dumps({
    'resume_step': identity.global_step,
    'stage': identity.training_stage,
    'dense_params': params,
    'parameter_sha256': (identity.parameter_sha256 or '')[:16],
}, indent=2))
del model

In [ ]:
# 7. Train. WSD schedule decays LR across TOKEN_BUDGET so the run ends at
# the pack boundary instead of grinding repeat passes at full learning rate.
import sys
sys.path.insert(0, str(REPO))
os.chdir(str(REPO))

from training.train_tpu import parse_args, run

argv = [
    '--pack-root', PACK_DIR,
    '--resume-from', RESUME_FROM,
    '--output-checkpoint', OUTPUT_CKPT,
    '--max-minutes', str(MAX_MINUTES),
    '--token-budget', str(TOKEN_BUDGET),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum-steps', str(GRAD_ACCUM),
    '--learning-rate', str(LEARNING_RATE),
    '--min-lr-ratio', str(MIN_LR_RATIO),
    '--save-interval', str(SAVE_INTERVAL),
    '--log-interval', str(LOG_INTERVAL),
    '--seed', str(SEED),
]
exit_code = run(parse_args(argv))
print(f'training exit code: {exit_code}')
assert exit_code == 0, 'training failed - inspect log above'

In [ ]:
# 8. Post-session capability probe: did this session move the substrate?
# These are the G-B floor probes from POST_20K_AGI_PATH.md.
import json
from connector.experiments.cognitive_credit.capability_probe import run_probe
probe = run_probe(OUTPUT_CKPT, device='xla')
print(json.dumps(probe, indent=2))
floor = ['P1_knowledge_use', 'P2_plan_following', 'P3_verbatim_echo', 'P4_tool_result_use']
failed = [name for name in floor if int(probe[name].split('/')[0]) < 4]
print('SUBSTRATE FLOOR:', 'PASSED - Phase S (SFT) may proceed' if not failed
      else f'NOT PASSED yet ({", ".join(failed)}) - continue Phase B with fresh packs')

In [ ]:
# 9. Persist artifacts: one canonical output + a session metrics receipt.
import shutil, time
RECEIPT = {
    'source_commit': SOURCE_COMMIT,
    'resume_from_sha256': sha256_file(Path(RESUME_FROM))[:16],
    'output_sha256': sha256_file(Path(OUTPUT_CKPT)),
    'pack_root': PACK_DIR,
    'probe': probe,
    'finished_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
Path('/kaggle/working/session_receipt.json').write_text(json.dumps(RECEIPT, indent=2))
shutil.copy('/kaggle/working/session_receipt.json', '/kaggle/working/metrics.json')
print('Upload BOTH files to the Drive vault:')
print(' ', OUTPUT_CKPT)
print('  /kaggle/working/session_receipt.json')